# The Möller-Wedel Fizeau Interferometer

## Introduction

This experiment gives you experience using a Fizeau interferometer and digital phase analysis. Such instruments are often called Zygos, after the instruments manufactured by Zygo Inc. The instrument here is manufactured by Möller-Wedel. Acquisition and analysis use **MWdriver**, the software in this repository. The laboratory uses a Thorlabs digital camera and a Raspberry Pi Pico to coordinate phase-shifter drive timing and six hardware camera triggers.

The Möller-Wedel Fizeau differs from the Twyman-Green interferometer in the demonstration experiments in three important respects: it has a piezo-driven phase-shifting reference surface at the output aperture; interferograms are recorded digitally rather than viewed only by eye; and it uses a HeNe laser. The reference surface lets you measure the returned test wavefront relative to that surface. Moving it changes the optical path difference (OPD), allowing the computer to determine both the magnitude and sign of the wavefront from a sequence of intensity images. The laser relaxes restrictions on the OPD between test and reference beams.

Follow the alignment, calibration and measurement stages carefully. Distinguish measured wavefront OPD from the surface height of a reflecting optic and from single-pass aberrations of a double-pass optical system. Check the units and correction stages of each MWdriver output before interpreting it.

<div class="alert alert-block alert-info"> For your report, concentrate on the measurements in Exercises 1 and 2, including calibration quality, repeatability and the interpretation of the fitted aberrations. </div>

## Setting up the Interferometer

This section introduces the optical instrument and its controls. The instrument manuals on the shelf behind the bench remain useful for the optics, FINDER, FOCUS and ZOOM controls. Use the repository [README](../README.md) and this script for MWdriver. Reference manuals must remain in the room.

The interferometer should already be switched on and warmed up. If not, ask the demonstrator to confirm the correct power-up sequence for the interferometer, laser, installed camera and phase-shifter controller. Initialise the Mitutoyo axis counter using the appropriate reset button. Follow the laboratory laser-safety rules and do not look into the beam.

Use the Thorlabs **Camera Preview** in MWdriver to check the image that will be recorded. Start the software and preview as described in the next section; keep hands away from electrical connections and ask the demonstrator if the alignment display differs from the instrument manual. If an instrument monitor is connected, it may also assist alignment.

Press FINDER on the instrument's remote control to enter autocollimating mode. Returning collimated beams are focused to spots: one from the reference flat and another from the test mirror. If fitted, the instrument monitor's cross provides an alignment reference. The internal mechanism may require another press to seat correctly. Adjust the reference-flat tilt controls to centre its spot. Mount the circular uncoated mirror in the scroll chuck of the five-axis mount, polished side facing the interferometer. Centre it approximately in the output beam and adjust tilt until the return spot coincides with the reference spot. Press FINDER again to return to fringe viewing. If you cannot obtain fringes, ask a demonstrator for help.

Use the five-axis mount tilt controls to practise changing the fringe direction and spacing; use its translation controls to centre the pupil. Use FOCUS to obtain a clear pupil edge and ZOOM to fill the camera field without clipping it. Investigate LASER and IRIS while watching the live camera image. What happens at excessive settings, and why? Avoid saturating the recorded camera intensity.

A single-longitudinal-mode HeNe laser can have a long coherence length, but this instrument operates in several longitudinal modes. As you move the mount along the slide, you should find positions with higher and lower fringe contrast. Investigate how these positions relate to the laser cavity length and how contrast changes during warm-up. Choose a high-contrast position for measurements.

The fringes map OPD between the reference flat (specified as flat to $\lambda/20$) and the test mirror. Draw a schematic of the external and internal optics, using the instrument manual. Adjacent fringes differ by $\lambda$ in OPD; a deviation of one fringe spacing therefore represents $\lambda/2$ in mirror surface height. Explain the factor of two and what limits accuracy, including the reference surface, vibration and camera signal quality.

MWdriver saves each measurement as a six-frame NumPy stack with PNG images and metadata; analysis adds numerical and graphical results. Keep these files together and copy the complete session folders to your group's approved storage or USB device. Screenshots are useful illustrations, but do not replace the raw data. The appendix explains how to load these files.

## Software: MWdriver

The experiment uses **MWdriver**, a Python application with a Napari/Qt viewer, a Thorlabs digital camera and a Raspberry Pi Pico phase-shifter controller. Use the repository [README](../README.md) for software information and troubleshooting. The [MWdriver source code on GitHub](https://github.com/ImperialCollegeLondon/MWdriver) is available for reference; laboratory users should run the installed application, not the source checkout. Acquisition requires the supported Windows camera drivers, official Thorlabs SDK and Pico firmware, not just the Python packages.

1. Ask the demonstrator to confirm that the camera, Pico and phase-shifter drive circuitry are connected and powered. The Pico must run `pypico/main.py` under MicroPython. Its GP15 output is PWM and requires the appropriate drive/interface circuitry; do not connect it directly as an analogue piezo drive or alter the wiring yourself.
2. Close ThorCam and any Pico serial monitor before opening MWdriver. Launch **MWdriver** from the shared Start menu shortcut, or run `C:\Program Files\MWdriver\MWdriver.exe`. Use this installed version for all laboratory work.

   If launch fails, ask the demonstrator to check the installation, native DLLs and Pico serial-port setting. Do not change a working laboratory configuration without agreement.
3. The layer list is on the left; **MWdriver Controls** is a separate scrollable dock on the right. Choose **Camera > Start Stream** and keep the **Camera Preview** layer visible to view live fringes. **Camera > Return to Camera Preview** restores preview after inspecting a capture or analysis. **Stop Stream** disables live acquisition.
4. Focus the pupil edge, centre it in the image and use the optical zoom to fill the field without clipping the pupil. Adjust the camera exposure and available gain controls, together with LASER and IRIS, for clear modulation without saturation. Inspect the intensity values in the preview, not only the display brightness: Napari contrast limits change the display, not the recorded signal. Explain why saturation, insufficient signal and poor fringe contrast compromise phase measurements.
5. Select **ROI > Ellipse ROI** and draw an ellipse around the useful pupil. Adjust its position and size to exclude the background and unreliable pupil edge; for a circular pupil keep it circular. Pixels outside this single ellipse are masked automatically. Use **ROI > Save ROI** to persist it and **ROI > Reload ROI** to restore the last separately saved ellipse. Redraw it whenever the pupil changes. This analysis ROI selects the useful pupil; it does not crop the camera sensor.

If no ROI is defined, analysis can derive a circular pupil from valid data, but inspect that choice before using the results. Save the ROI once it matches the physical pupil. Keep the same physical pupil when comparing lens orientations, since fitted aberrations depend on pupil size.

### Calibrating the phase steps

Arrange approximately ten to twenty clear, nearly straight fringes (certainly avoid any loop fringes) across the pupil and check exposure and ROI. In the Camera Status controls select **90 degree six-frame**, the default method for this experiment. Its six target positions are $0,90,180,270,360,450^\circ$: six images span five intervals. The alternative **60 degree six-bucket** method targets $0,60,120,180,240,300^\circ$ and requires its own calibration. Record the chosen method; do not change it partway through a comparison without recalibrating.

Enable **Analysis > Extra Diagnostic Plots** before calibrating if you want the combined iteration curves, errors and Fourier/field maps for your report. This session option is off by default. Select **Camera > Recalibrate**, stand back from the bench and remain still throughout all captures. The Pico drives the phase shifter and triggers six camera exposures; the software measures relative phase changes and adjusts trigger timings along the drive waveform. Calibrating these timings does not replace the instrument's mechanical alignment.

Recalibration begins with a coarse capture and then verification captures. Two consecutive captures within $\pm5^\circ$ confirm calibration. Up to six verification captures are allowed before a restart decision, with at most one automatic restart. Crucially, a completed run at the limit retains the final capture even if it has not passed. Read the status and inspect `measured_vs_expected_phase_annotated.png` and `measured_phase_check.json`; do not treat the existence of a saved calibration as proof of success. With extra diagnostics enabled, inspect `recalibration_iterations.png` and `recalibration_iteration_errors.png` to discuss convergence.

Why do vibration, saturation, weak fringes or phase-shifter non-linearity affect the phase check? How do the measured increments compare with the target? Repeat calibration after changes to the optical setup or selected phase-step method, or if subsequent checks show poor agreement. Ask the demonstrator for help if it fails; do not deliberately disturb the bench. Keep the camera in an unsaturated operating range.

### Saving and identifying your data

MWdriver automatically saves user data in the laboratory account's actual **Documents/MWDriver** folder, including redirected or OneDrive Documents locations.

- Measurements use `measure_YYYYMMDD_HHMMSS`; calibrations use `calibrate_YYYYMMDD_HHMMSS`.
- Each measurement includes one `*_raw_stack.npy`, six PNG intensity frames and `*_metadata.json`, together with the automatic phase-check JSON and plot.
- GUI analysis adds `phase_analysis_results.npz`, intermediate images and `phase_analysis_result_images/`.
- The separately saved `roi_params.json`, `roi_mask.npy` and `phase_trigger_positions.json` are in Documents/MWDriver, outside the source tree. Copy the relevant settings alongside your experiment record.

In your lab notes, associate each timestamped folder with your group, optic, lens orientation, pupil diameter, slide position, phase-step method, exposure and calibration outcome. Copy complete measurement and relevant calibration folders to approved group storage before leaving; do not move or overwrite another group's files. The final verification calibration is retained, while coarse and rejected verification folders are normally removed. Retain its diagnostic plots if enabled.

### Acquiring and analysing a measurement

1. Check alignment, pupil ROI, exposure, phase-step method and calibration. Stand back and remain still, then choose **Start Phase Capture**. Preview redraws pause while six externally triggered frames are collected. Wait for saving and the automatic phase check to finish before another capture.
2. Inspect the saved measured-versus-expected plot. Why are six intensity images needed to recover phase rather than one? Why are there five phase intervals? Discuss any departures from the target and repeat an unreliable measurement only after addressing its cause.
3. The completed raw stack appears as a separate `time, y, x` layer. Select it and move the time slider through frames 0–5 to inspect the fringes. These are the original intensity images, not reconstructed phase. Use **Camera > Return to Camera Preview** for live alignment.
4. Select the intended stack and choose **Analysis > Analyze Selected Stack**. **Analyze Latest Capture** instead uses the latest saved measurement, which may not be the run you meant. Analysis uses the capture's recorded phase-step method and the current ROI. Check the configured wavelength with the demonstrator: the default is 632.8 nm for this HeNe instrument.
5. Select the new analysis layer and use its review selector to inspect wrapped phase, unwrapped phase, tilt-removed results, coefficient charts and wavefront results. Wrapped and unwrapped images display waves; tilt-removed and wavefront views display nanometres of OPD. Stored phase arrays are in radians. Compare the reconstructed structure with the fringes you observed and explain wrapping and unwrapping.
6. Compare **Interactive 3D: Tilt removed**, **Interactive 3D: Tilt and defocus removed** and **Interactive 3D: Residual wavefront**. Drag to rotate and scroll to zoom. The saved `wavefront_surfaces.png` compares these stages with a common OPD scale. Distinguish the removal of alignment terms from removal of real aberrations: a small residual does not by itself imply a good optic.

To reopen a measurement folder, choose **Analysis > Reload Dataset...** and select its `measure_...` directory. MWdriver restores the raw stack and metadata and loads saved analysis if the results archive is present; otherwise, select the restored stack and choose **Analyze Selected Stack**. To load one raw stack file, choose **Analysis > Reload Stack...**, then select its layer and analyse it as needed. To review saved results without recalculating, choose **Analysis > Reload Analysis...** and select the `phase_analysis_results.npz` file. All reload actions are in the **Analysis** menu. Reloading does not change camera settings or the current ROI; record which ROI was used for each recalculated result.

Repeat the mirror measurement to estimate repeatability before moving to the lenses. Record results and uncertainties in the following cells.

<div class="alert alert-block alert-success"> Add your own input here. </div>

### Initialisation procedure

At every launch, confirm the camera and Pico connection, start the visible Camera Preview, check exposure and gain, and draw or reload the pupil ROI. Verify the selected phase-step method and recalibrate for the current setup. Saved ROI and trigger timing files are available across sessions, but are not proof that today's alignment and calibration are suitable. Raw intensities are saved automatically. Enable **Analysis > Extra Diagnostic Plots** again if needed: it is a session setting, off by default.

### Switch off

Wait for capture, calibration, saving and analysis to finish. Copy your complete datasets and note their folder names. Choose **Camera > Stop Stream**, then close the MWdriver/Napari window normally before shutting down the computer or disconnecting hardware. Ask the demonstrator for the laboratory's laser, interferometer and controller shutdown procedure; do not interrupt a running acquisition or alter wiring.

Before continuing, resolve any problems with the demonstrator and retain one reliable mirror measurement as a reference.

## Exercise 1. To check the figure and curvature of the surfaces of a plano-convex lens

You have two plano-convex lenses marked f50 and f200, their nominal focal lengths in millimetres. Check the figure and radius of curvature of both surfaces of the f50 lens, then compare with the f200 lens.

First measure the plano surface of each lens using the mirror workflow above. Redraw and save the pupil ROI, check exposure and recalibrate for the new setup. Save repeated captures and analyse each selected stack. Inspect the **before-correction** Zernike chart, especially signed **J4 Defocus**, before using corrected views. From this defocus and a measured physical pupil radius, can you estimate an upper limit on the surface radius of curvature? Include uncertainty and distinguish wavefront OPD from reflecting-surface height. Are other aberrations significant?  Does the planar surface look flat even ?  Some lenses are better than others and you should choose the best that you can find!

A curved surface requires a converger rather than the flat reference. This is a special positive lens giving a converging output beam with an appropriate reference surface. The reference is the outer surface: explain why. Do not touch it; contamination changes reflectivity and phase, distorts fringes and may require cleaning that risks coating wear.

Ask a demonstrator to replace the flat reference with the 120 mm converger. Do not change it yourself. Mount the f50 lens with its curved side towards the interferometer. Use FINDER and a piece of paper to locate the return beam; adjust position and orientation to make it retrace its path. You can find straight fringes at two positions:

1. **Position A:** the converger focus is on the lens surface.
2. **Position B:** the converger focus is at the centre of curvature of that surface.

Slide the mount to find both. At A, the reference-surface alignment is especially sensitive, and its tilt may need adjustment. Ask for help if necessary. At B, the fringes compare the curved surface figure with the converger reference. Explain what is measured in each position. The axial travel between A and B, read on the Mitutoyo counter, measures the radius of curvature. Zero the counter at one position using the appropriate reset button and repeat the travel measurement to assess uncertainty; the other counter functions are not needed on this slide.

Use the radius to estimate focal length for Schott BK7 at the interferometer wavelength, 632.8 nm. Research the refractive index and the manufacturer's specification. What limits accuracy? A singlet's focal length depends on wavelength, so the result need not equal exactly 50 mm; state any thin-lens approximation and whether thickness matters.

Two other strong return positions may be found, not necessarily with straight fringes:

3. **Position C:** the converger focuses through the front surface onto the back surface.
4. **Position D:** the lens collimates the divergent wave after the converger focus, and it reflects from the back plano surface.

Why can the A-to-D distance provide another focal-length estimate? Keep the central fringes as straight as possible when seeking the paraxial focal length.

Repeat with the f200 lens, taking care near the reference surface. Return to B and measure its curved-surface figure with MWdriver: update the ROI, recalibrate, capture and choose **Analysis > Analyze Selected Stack**. Compare the tilt-removed view with **Interactive 3D: Tilt and defocus removed** and the saved `wavefront_aberration_nm` map. MWdriver removes piston, tilt and defocus automatically for this wavefront-aberration result. Record the before-correction J4 coefficient as well as corrected RMS/PV. Do not use the further-corrected residual to claim absence of astigmatism or spherical aberration, since those terms have been removed from it.

For reflection at near-normal incidence, surface height is half the measured wavefront OPD. State this convention and the role of reference-surface error in your reported figure.

<div class="alert alert-block alert-success"> Add your own input here. </div>

## Exercise 2. To build and test a ×4 magnification afocal beam expander

Ask the demonstrator to replace the converger with the flat $\lambda/20$ reference. Return the uncoated mirror to the five-axis mount, obtain fringes and minimise their number. Although the mirror is not perfectly flat, the approximately 5 mm diameter area used here should be much closer to flat.

Mount the black two-lens jig on the blue table between the interferometer and mirror. Place the 200 mm lens nearest the interferometer and the 50 mm lens further away. Why is the expander used in this direction during the test? Adjust lateral position, tilt and lens separation so the reduced output beam is collimated and returns from the mirror through both lenses. Use paper to locate the foci and FINDER for alignment; confirm that you are seeing the mirror return rather than an unwanted lens-surface reflection.

Focus the image of the limiting pupil, the 200 mm lens aperture. You observe it in double pass, so there are potentially two pupil images. Where should the mirror be placed to superimpose them?

<div class="alert alert-block alert-info"> Hint: if the first pass images the first-lens pupil onto the mirror, the second pass images it back onto that lens. </div>

Adjust the system for approximately four or five straight fringes. Draw the useful pupil ROI in MWdriver, check exposure and recalibrate. Capture and analyse repeated datasets as for the mirror. For each of the four lens-orientation combinations, record the orientation of both curved surfaces, pupil diameter, separation, calibration outcome, signed **J11 Primary Spherical**, other significant J1–J11 coefficients, and wavefront RMS/PV after piston, tilt and defocus removal. Keep the same physical pupil and measurement conditions for comparisons. Use the before-correction coefficient chart or exported dictionary, not only the absolute-magnitude composition chart or the residual after spherical removal.

Seidel theory gives the spherical aberration introduced by a singlet as

$$S_I=\frac{h^4K^3}{4}\left[\left(\frac{n}{n-1}\right)^2+\frac{n+2}{n(n-1)^2}\left(B+\frac{2(n^2-1)}{n+2}C\right)^2-\frac{n}{n+2}C^2\right],$$

where $h$ is marginal ray height, $K$ is lens power, and

$$B=\frac{c_1+c_2}{c_1-c_2},\qquad C=\frac{u_1+u_2'}{u_1-u_2'}.$$

For a plano-convex lens, $B=+1$ or $-1$ according to orientation; $C=+1$ or $-1$ depends on which conjugate is at infinity. State your sign conventions. Predict the optimum combination and compare all four measurements with theory. Which lens is more sensitive to orientation? Remember that the two lenses have different marginal ray heights and that $S_I$ scales with $h^4$.

### Comparing Seidel theory with MWdriver

The single-pass Seidel wavefront term is

$$W_{\rm single}(\rho)=\frac{S_I}{8}\rho^4,\qquad 0\leq\rho\leq1.$$

MWdriver reports signed, unit-normalised Zernike coefficients in **waves of measured wavefront OPD**. Its primary spherical polynomial is

$$Z_{11}(\rho)=\sqrt5\,(6\rho^4-6\rho^2+1).$$

If $W_{\rm single}/\lambda=A_{\rm single}\rho^4$, and the reciprocal, well-aligned double pass contributes twice the single-pass aberration, then

$$J_{11,\rm measured}=\frac{2A_{\rm single}}{6\sqrt5},\qquad A_{\rm single}=3\sqrt5\,J_{11,\rm measured}.$$

For the two lenses, sum their predicted $S_I/8$ contributions using each lens's own $h$ and consistent length units before dividing by $\lambda$. Derive the Zernike conversion by expressing $\rho^4$ in piston, defocus and primary spherical modes. Do not directly compare J11 with $S_I/(8\lambda)$, and do not assume MWdriver has already converted all outputs to single pass. Discuss deviations due to alignment, unequal forward/return pupils, reference and mirror errors, higher-order aberrations and finite pupil sampling. The archive's `residual_nm` uses a reflecting-surface-height convention; it is not the beam expander's single-pass wavefront map.

<div class="alert alert-block alert-success"> Add your own input here. </div>

## Appendix: Importing and interpreting MWdriver data

Use a complete `measure_YYYYMMDD_HHMMSS` folder produced by MWdriver. In the installed application, load it from **Analysis > Reload Dataset...**. If the folder has no saved analysis, select the restored raw-stack layer and choose **Analysis > Analyze Selected Stack**; the application saves `phase_analysis_results.npz` in the session folder. To review an existing analysis without recalculating it, choose **Analysis > Reload Analysis...** and select that NPZ file. All dataset and analysis reloads are performed in the application's **Analysis** menu; no direct access to analysis scripts is needed.

The following notebook cells inspect files from a dataset that has already been captured and analysed. Set `session` in the loading cell to the folder path on this computer, then run the appendix cells in order. These cells need NumPy and Matplotlib, not the camera SDK or attached hardware. Loading is read-only. Phase arrays in the NPZ are in radians, excluded pixels are NaN, and `roi_mask=True` means inside the spatial ROI. The signal-valid analysis mask may be smaller. Coefficient dictionaries are JSON strings, so use `allow_pickle=False`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
session = Path(r"C:\path\to\Documents\MWDriver\measure_YYYYMMDD_HHMMSS")
if not session.is_dir():
    raise FileNotFoundError("Set session to your copied MWdriver measurement directory.")

stack_paths = list(session.glob("*_raw_stack.npy"))
if len(stack_paths) != 1:
    raise ValueError("Choose a dataset folder containing exactly one raw stack.")
stack_path = stack_paths[0]
metadata_path = stack_path.with_name(
    stack_path.name.removesuffix("_raw_stack.npy") + "_metadata.json"
)
if not metadata_path.is_file():
    raise FileNotFoundError("Keep the capture metadata JSON beside the raw stack.")
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
with np.load(session / "phase_analysis_results.npz", allow_pickle=False) as archive:
    results = {name: archive[name] for name in archive.files}

wavelength_nm = float(results["wavelength_nm"].item())
roi_mask = results["roi_mask"].astype(bool)
wrapped_saved = np.ma.masked_invalid(results["wrapped"])
unwrapped_saved = np.ma.masked_invalid(results["unwrapped"])
phaseT = np.ma.masked_invalid(results["tilt_removed"])
wavefront_nm = np.ma.masked_invalid(results["wavefront_aberration_nm"])
residual_surface_nm = np.ma.masked_invalid(results["residual_nm"])
coefficients_before = json.loads(
    results["zernike_coefficients_before_correction_waves"].item()
)
coefficients_after = json.loads(
    results["zernike_coefficients_after_correction_waves"].item()
)
algorithm_id = metadata.get("phase_capture_model", {}).get(
    "algorithm_id", "six_frame_90deg"
)
if algorithm_id != str(results["algorithm_id"].item()):
    raise ValueError("Capture and analysis use different phase-step methods.")
print(f"Dataset: {session.name}; method: {algorithm_id}; wavelength: {wavelength_nm:g} nm")

### Inspect the six raw intensity frames

The raw NumPy stack has shape `(6, y, x)`. Frame indices run from 0 to 5. Use a shared display scale to compare intensity and fringe displacement; changing a plot's limits does not change the recorded data. The capture metadata records the selected method and acquisition settings. Check for saturation, motion and weak modulation before interpreting a reconstructed surface.

In [ ]:
data = np.asarray(np.load(stack_path, allow_pickle=False), dtype=np.float64)
if data.ndim != 3 or data.shape[0] != 6:
    raise ValueError(f"Expected a (6, y, x) stack, got {data.shape}.")
if roi_mask.shape != data.shape[1:] or wrapped_saved.shape != data.shape[1:]:
    raise ValueError("Capture, ROI and saved phase dimensions do not match.")

fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)
for frame_index, axis in enumerate(axes.flat):
    image = axis.imshow(data[frame_index], cmap="gray", vmin=data.min(), vmax=data.max())
    axis.set_title(f"Frame {frame_index}")
    axis.set_axis_off()
fig.colorbar(image, ax=axes.ravel().tolist(), label="Camera intensity (recorded units)")
plt.show()

### Recover wrapped phase using the recorded method

For the default 90-degree method, MWdriver uses the six-frame averaging formula with numerator $4I_3-3I_1-I_5$ and denominator $I_0-4I_2+3I_4$. Use this sign convention consistently when comparing reconstructed and saved phase. For 60-degree captures, use the six-bucket sine/cosine sums instead. Six images correspond to five intervals, not six steps between images. Compare with the phase-shifting discussion in [Optical Shop Testing by Daniel Malacara](https://library-search.imperial.ac.uk/permalink/44IMP_INST/f7tnsv/alma991000343771501591).

For a fair comparison, the cell below keeps only pixels that are finite in the saved wrapped-phase result. That result already includes MWdriver's spatial ROI and PSI signal-validity filtering; the Boolean spatial ROI alone does not describe all valid pixels.

In [ ]:
if algorithm_id == "six_frame_90deg":
    numerator = 4 * data[3] - 3 * data[1] - data[5]
    denominator = data[0] - 4 * data[2] + 3 * data[4]
elif algorithm_id == "six_bucket_60deg":
    phase_steps = np.deg2rad(np.arange(6) * 60.0)
    numerator = -np.tensordot(np.sin(phase_steps), data, axes=(0, 0))
    denominator = np.tensordot(np.cos(phase_steps), data, axes=(0, 0))
else:
    raise ValueError(f"Unsupported phase-step method: {algorithm_id}")

valid_mask = roi_mask & np.isfinite(results["wrapped"])
if not np.any(valid_mask):
    raise ValueError("The saved analysis contains no valid phase pixels.")
phase = np.ma.masked_array(np.arctan2(numerator, denominator), mask=~valid_mask)
fig, axis = plt.subplots()
image = axis.imshow(phase / (2 * np.pi), cmap="twilight", vmin=-0.5, vmax=0.5)
fig.colorbar(image, ax=axis, label="Wrapped phase (waves)")
plt.show()

### Compare wrapped phase and inspect unwrapping

Subtract phases modulo $2\pi$ so a wrapping boundary does not appear as a large error. Agreement here checks the reconstruction formula and sign, not the physical accuracy of the instrument. Inspect MWdriver's saved `unwrapped` array rather than applying a row/column unwrap across masked background. Discuss why unwrapping becomes unreliable where fringe modulation is weak or valid regions are disconnected.

In [ ]:
phase_difference = np.ma.array(
    np.angle(np.exp(1j * (phase - wrapped_saved))), mask=~valid_mask
)
print(f"Maximum wrapped disagreement: {np.ma.max(np.ma.abs(phase_difference)):.3g} rad")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for axis, values, title, units in [
    (axes[0], phase_difference, "Reconstruction minus saved phase", "Phase difference (rad)"),
    (axes[1], unwrapped_saved / (2 * np.pi), "Saved unwrapped phase", "Phase (waves)"),
]:
    image = axis.imshow(values)
    axis.set_title(title)
    fig.colorbar(image, ax=axis, label=units)
plt.show()

### Check the analysis pupil

Compare the saved spatial ROI with the finite-data mask. MWdriver fits J1–J11 simultaneously by least squares using its pupil geometry; it does not assume exact orthogonality of simple pixel averages. A partial or poorly chosen pupil changes the fitted coefficients. The pixel centroid below is only a diagnostic, not a replacement for the application's ellipse geometry. Measure and record the physical pupil radius separately for curvature and Seidel calculations.

In [ ]:
inside_y, inside_x = np.nonzero(roi_mask)
print(f"Spatial ROI pixels: {inside_x.size}; valid phase pixels: {valid_mask.sum()}")
print(f"Spatial ROI pixel centroid: x={inside_x.mean():.2f}, y={inside_y.mean():.2f}")
fig, axis = plt.subplots()
axis.imshow(roi_mask, cmap="gray", vmin=0, vmax=1)
axis.contour(valid_mask.astype(float), levels=[0.5], colors=["red"])
axis.set_title("Spatial ROI with valid-phase boundary")
plt.show()

### Compare correction stages in wavefront OPD

Stored `tilt_removed` is a phase array in radians after piston and tilt removal. Convert it to wavefront OPD using $W=\phi\lambda/(2\pi)$. The `wavefront_aberration_nm` array is already in nanometres and additionally removes defocus. Use common colour limits to compare these two maps. Keep the signed J4 coefficient for curvature estimates before discarding defocus.

In [ ]:
tilt_removed_opd_nm = phaseT * wavelength_nm / (2 * np.pi)
comparison_values = np.concatenate([
    tilt_removed_opd_nm.compressed(), wavefront_nm.compressed()
])
colour_min, colour_max = np.percentile(comparison_values, [0.1, 99.1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for axis, values, title in [
    (axes[0], tilt_removed_opd_nm, "Piston and tilt removed"),
    (axes[1], wavefront_nm, "Piston, tilt and defocus removed"),
]:
    image = axis.imshow(values, cmap="viridis", vmin=colour_min, vmax=colour_max)
    axis.set_title(title)
fig.colorbar(image, ax=axes.tolist(), label="Wavefront OPD (nm)")
plt.show()

### Report wavefront figure and repeatability

The exported RMS and robust PV refer to `wavefront_aberration_nm`, after piston, tilt and defocus removal. Robust PV is the 99.9th minus 0.1st percentile, not the absolute maximum minus minimum. Compare repeated runs using the same physical pupil and correction stage, and include calibration quality and reference-surface error in your uncertainty discussion.

In [ ]:
rms_nm = float(results["wavefront_aberration_rms_nm"].item())
pv_nm = float(results["wavefront_aberration_pv_nm"].item())
print(f"Piston, tilt and defocus removed: RMS={rms_nm:.3f} nm OPD")
print(f"Robust PV (99.9th minus 0.1st percentile)={pv_nm:.3f} nm OPD")
print(f"Reflecting-surface equivalents: RMS={rms_nm / 2:.3f} nm; PV={pv_nm / 2:.3f} nm")

### Read signed Zernike coefficients

Print the before- and after-correction dictionaries, in waves of measured wavefront OPD at the exported wavelength. J1–J11 end at primary spherical. The composition chart instead reports non-negative combined magnitudes for astigmatism, coma, trefoil and spherical after piston, tilt and defocus removal. For Exercise 2, use signed J11 and the Seidel/Zernike and double-pass conversion derived there; the sign depends on the instrument and phase convention. For Exercise 1, use before-correction J4 for defocus.

In [ ]:
print("Signed coefficients in waves of measured wavefront OPD:")
for name, value in coefficients_before.items():
    print(f"{name}: before={value:+.6f}; after={coefficients_after[name]:+.6f}")

j11_measured_waves = float(coefficients_before["J11 Primary Spherical"])
single_pass_seidel_edge_waves = 3 * np.sqrt(5) * j11_measured_waves
print(f"Measured J11: {j11_measured_waves:+.6f} waves")
print(
    "Equivalent single-pass Seidel rho^4 edge amplitude "
    f"(ideal reciprocal double pass): {single_pass_seidel_edge_waves:+.6f} waves"
)

### Surface height and the further-corrected residual

For reflection at near-normal incidence, one radian of measured phase corresponds to surface height $\lambda/(4\pi)$, approximately 50.4 nm at 632.8 nm. This is half the wavefront OPD per radian. For a double-pass transmissive system, interpret the single-pass wavefront separately, as in Exercise 2.

MWdriver's `residual_nm` already uses the reflecting-surface-height convention and removes piston, tilt, defocus, astigmatism and primary spherical. Do not divide it by two again. The application's 3D residual-wavefront view multiplies this array by two to put it on the same OPD scale as the other surfaces. Coma, trefoil, higher-order structure, noise and reference errors can remain. Compare this residual with the earlier corrected wavefront and explain which genuine aberrations have been removed.

In [ ]:
print(f"Surface height per radian at {wavelength_nm:g} nm: {wavelength_nm / (4 * np.pi):.3f} nm")
residual_opd_nm = 2 * residual_surface_nm
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for axis, values, title, units in [
    (axes[0], residual_surface_nm, "Further-corrected residual: surface convention", "Equivalent surface height (nm)"),
    (axes[1], residual_opd_nm, "Same residual: wavefront convention", "Wavefront OPD (nm)"),
]:
    image = axis.imshow(values, cmap="viridis")
    axis.set_title(title, fontsize=10)
    fig.colorbar(image, ax=axis, label=units)
plt.show()